# Generation Evaluation

Evaluates the outputs of the RAG inference pipeline (`inference.ipynb`) against gold answers from the benchmark dataset.

**Metrics implemented**

| Category | Metric | Notes |
|---|---|---|
| Lexical overlap | Exact Match (EM) | Normalised string equality |
| Lexical overlap | Token-level F1 | Precision/Recall over shared tokens |
| Lexical overlap | ROUGE-1 / ROUGE-2 / ROUGE-L | Unigram, bigram, longest common subsequence |
| Lexical overlap | BLEU-1 / BLEU-4 | 1-gram and 4-gram precision with brevity penalty |
| Semantic similarity | BERTScore (F1) | Contextual embedding cosine similarity |
| Breakdown | By source corpus | Wikipedia / Wikibooks / Blog |

**Input files**
- `results/output_results.json` — single-run inference output (or any file matching the schema)
- `../data/benchmark_dataset.json` — gold answers
- *(optional)* `results/exploration/output_*.json` — exploration grid outputs


In [1]:
%pip install rouge-score bert-score --quiet


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, os, re, string
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Tuple

try:
    from rouge_score import rouge_scorer as _rs
    ROUGE_AVAILABLE = True
except ImportError:
    ROUGE_AVAILABLE = False
    print("[WARN] rouge-score not installed. Run: pip install rouge-score")

try:
    from bert_score import score as _bert_score
    BERTSCORE_AVAILABLE = True
except ImportError:
    BERTSCORE_AVAILABLE = False
    print("[INFO] bert-score not installed (optional). Run: pip install bert-score")

print(f"ROUGE      : {'available' if ROUGE_AVAILABLE else 'NOT available'}")
print(f"BERTScore  : {'available' if BERTSCORE_AVAILABLE else 'NOT available'}")


d:\app\miniconda\envs\rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ROUGE      : available
BERTScore  : available


In [3]:
BENCHMARK_PATH  = "../data/benchmark_dataset.json"
OUTPUT_PATH     = "results/output_results.json"   # main output to evaluate
EXPLORATION_DIR = "results/exploration"            # set None to skip batch eval
EVAL_OUTPUT     = "results/generation_evaluation_summary.json"

# BERTScore model — distilbert is faster; use bert-base-uncased for higher quality
BERTSCORE_MODEL = "distilbert-base-uncased"

# Response truncation: extract only the first N sentences / M words
# This reduces the precision penalty from verbose generation
TRUNCATE_SENTENCES = 3   # take first 3 sentences of model output
TRUNCATE_WORDS     = 80  # hard cap at 80 words

os.makedirs("results", exist_ok=True)


## 1. Load Benchmark

In [4]:
def load_benchmark(path: str) -> Dict[str, dict]:
    """Load nested benchmark JSON. Returns dict keyed by q_001, q_002, ..."""    
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    benchmark = {}
    idx = 1
    for src_block in data["sources"]:
        source = src_block["source"]
        for item in src_block["questions"]:
            qid = f"q_{idx:03d}"
            benchmark[qid] = {
                "query_id": qid,
                "query":    item["question"],
                "answer":   item["answer"],
                "source":   source,
            }
            idx += 1
    print(f"Loaded {len(benchmark)} benchmark entries")
    return benchmark

benchmark = load_benchmark(BENCHMARK_PATH)
for qid, e in list(benchmark.items())[:2]:
    print(f"  {qid}: {e['query'][:70]}")
    print(f"       → {e['answer'][:80]}")


Loaded 72 benchmark entries
  q_001: What does the word 'bibimbap' literally mean in Korean?
       → The term bibim means mixing and bap is cooked rice, meaning 'cooked rice', so it
  q_002: What does the Japanese term 'nuuhara' mean?
       → Nuuhara is short for 'nuudoru harasumento' (noodle harassment), describing the l


## 2. Metric Helpers

In [5]:
# ── Text normalisation ───────────────────────────────────────────────────────
def normalise(text: str) -> str:
    """Lowercase, strip articles and punctuation, collapse whitespace."""    
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def tokenise(text: str) -> List[str]:
    return normalise(text).split()


# ── Response truncation ──────────────────────────────────────────────────────
def truncate_response(text: str,
                      max_sentences: int = TRUNCATE_SENTENCES,
                      max_words: int = TRUNCATE_WORDS) -> str:
    """
    Extract the most informative part of a verbose model response.
    Takes the first max_sentences sentences (capped at max_words words).
    This reduces the precision penalty caused by long generative outputs.
    """
    text = text.strip()
    # Split on sentence-ending punctuation
    sentences = re.split(r"(?<=[.!?])\s+", text)
    truncated = " ".join(sentences[:max_sentences])
    words = truncated.split()
    if len(words) > max_words:
        truncated = " ".join(words[:max_words])
    return truncated


# ── Exact Match ──────────────────────────────────────────────────────────────
def exact_match(pred: str, ref: str) -> float:
    return float(normalise(pred) == normalise(ref))


# ── Token-level F1 / Precision / Recall ─────────────────────────────────────
def token_prf(pred: str, ref: str) -> Tuple[float, float, float]:
    """Returns (f1, precision, recall)."""    
    pt = tokenise(pred)
    rt = tokenise(ref)
    if not pt or not rt:
        return (1., 1., 1.) if pt == rt else (0., 0., 0.)
    common = sum((Counter(pt) & Counter(rt)).values())
    if common == 0:
        return 0., 0., 0.
    p = common / len(pt)
    r = common / len(rt)
    f1 = 2 * p * r / (p + r)
    return round(f1, 4), round(p, 4), round(r, 4)


# ── ROUGE ────────────────────────────────────────────────────────────────────
if ROUGE_AVAILABLE:
    _rouge = _rs.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=False)

def compute_rouge(pred: str, ref: str) -> dict:
    if not ROUGE_AVAILABLE:
        return {}
    s = _rouge.score(ref, pred)
    return {
        "rouge1_f": round(s["rouge1"].fmeasure, 4),
        "rouge1_r": round(s["rouge1"].recall,   4),
        "rouge2_f": round(s["rouge2"].fmeasure, 4),
        "rougeL_f": round(s["rougeL"].fmeasure, 4),
        "rougeL_r": round(s["rougeL"].recall,   4),
    }


# ── Smoke test ───────────────────────────────────────────────────────────────
pred = "Bibimbap literally means mixed rice in Korean where bibim refers to mixing and bap means rice."
ref  = "Bibimbap literally means mixed rice: bibim means mixing and bap means cooked rice."
f1, p, r = token_prf(pred, ref)
print(f"Token  F1={f1:.3f}  P={p:.3f}  R(Recall)={r:.3f}")
print(f"Trunc  words={len(truncate_response(pred*3).split())}")
if ROUGE_AVAILABLE:
    print("ROUGE:", compute_rouge(pred, ref))


Token  F1=0.759  P=0.688  R(Recall)=0.846
Trunc  words=46
ROUGE: {'rouge1_f': 0.7586, 'rouge1_r': 0.8462, 'rouge2_f': 0.5185, 'rougeL_f': 0.7586, 'rougeL_r': 0.8462}


## 3. Evaluation Functions

In [6]:
def evaluate_single(pred_raw: str, reference: str) -> dict:
    """
    Compute all metrics for one (prediction, reference) pair.
    Reports both raw (full response) and truncated metrics so the
    effect of response length on precision is visible.
    """
    pred_trunc = truncate_response(pred_raw)

    em = exact_match(pred_trunc, reference)

    # Truncated scores (primary — fairer for verbose models)
    f1_t, p_t, r_t = token_prf(pred_trunc, reference)
    # Full-response scores (secondary — shows raw verbosity effect)
    f1_r, p_r, r_r = token_prf(pred_raw, reference)

    result = {
        # Primary metrics (truncated response)
        "exact_match":      em,
        "f1":               f1_t,
        "precision":        p_t,
        "recall":           r_t,          # ← most meaningful for verbose output
        # Full-response token metrics
        "f1_full":          f1_r,
        "precision_full":   p_r,
        "recall_full":      r_r,
        # Response length info
        "pred_words":       len(pred_raw.split()),
        "pred_trunc_words": len(pred_trunc.split()),
        "ref_words":        len(reference.split()),
    }
    if ROUGE_AVAILABLE:
        rouge_t = compute_rouge(pred_trunc, reference)
        rouge_r = compute_rouge(pred_raw,   reference)
        result.update(rouge_t)
        # Keep full-response ROUGE-L recall separately
        result["rougeL_r_full"] = rouge_r.get("rougeL_r", 0.0)
    return result


def evaluate_output(output_path: str, benchmark: Dict[str, dict]) -> dict:
    """Evaluate one inference output JSON file against the benchmark."""    
    with open(output_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    results = data.get("results", [])

    per_query = []
    by_source = defaultdict(list)
    missing   = []

    for rec in results:
        qid = rec["query_id"]
        if qid not in benchmark:
            missing.append(qid)
            continue
        gold     = benchmark[qid]
        response = rec.get("response", "").strip()
        reference = gold["answer"].strip()

        scores = evaluate_single(response, reference)
        entry = {
            "query_id":  qid,
            "query":     rec["query"],
            "response":  response,
            "reference": reference,
            "source":    gold["source"],
            **scores,
        }
        per_query.append(entry)
        by_source[gold["source"]].append(scores)

    n = len(per_query)
    if n == 0:
        print(f"[WARN] No matching queries in {output_path}")
        return {}

    # Primary metrics to aggregate
    primary_keys = ["f1", "precision", "recall"]
    if ROUGE_AVAILABLE:
        primary_keys += ["rouge1_f", "rouge1_r", "rouge2_f", "rougeL_f", "rougeL_r"]
    all_keys = primary_keys + ["f1_full", "pred_words"]

    def avg(entries, key):
        vals = [e[key] for e in entries if key in e]
        return round(sum(vals) / len(vals), 4) if vals else 0.0

    aggregate = {k: avg(per_query, k) for k in all_keys}

    source_agg = {}
    for src, entries in by_source.items():
        source_agg[src] = {k: avg(entries, k) for k in primary_keys}
        source_agg[src]["n"] = len(entries)
        source_agg[src]["avg_pred_words"] = avg(entries, "pred_words")

    return {
        "output_path": output_path,
        "n_evaluated": n,
        "n_in_benchmark": len(benchmark),
        "n_missing": len(missing),
        "aggregate": aggregate,
        "by_source": source_agg,
        "per_query": per_query,
    }


def print_summary(tag: str, s: dict):
    agg = s["aggregate"]
    print(f"\n{'='*80}")
    print(f"  {tag}")
    print(f"{'='*80}")
    print(f"  Queries evaluated  : {s['n_evaluated']} / {s['n_in_benchmark']}")
    print(f"  Avg response length: {agg.get('pred_words',0):.0f} words")
    print()
    print(f"  ── Truncated response metrics (primary) ──────────────────────")
    print(f"  Token Recall       : {agg['recall']:.4f}   ← most reliable for verbose output")
    print(f"  Token F1           : {agg['f1']:.4f}")
    print(f"  Token Precision    : {agg['precision']:.4f}")
    if ROUGE_AVAILABLE:
        print(f"  ROUGE-1 F / R      : {agg.get('rouge1_f',0):.4f} / {agg.get('rouge1_r',0):.4f}")
        print(f"  ROUGE-L F / R      : {agg.get('rougeL_f',0):.4f} / {agg.get('rougeL_r',0):.4f}")
    print()
    print(f"  ── Full response metrics (for reference) ─────────────────────")
    print(f"  Token F1 (full)    : {agg.get('f1_full',0):.4f}  (lower due to verbosity)")
    if "bertscore_f" in agg:
        print(f"  BERTScore F1       : {agg['bertscore_f']:.4f}")
    print()
    print("  By source:")
    for src, stats in sorted(s["by_source"].items()):
        n = stats["n"]
        line = (f"    {src:40s} n={n:3d}  "
                f"Recall={stats['recall']:.4f}  F1={stats['f1']:.4f}")
        if ROUGE_AVAILABLE:
            line += f"  R1={stats.get('rouge1_f',0):.4f}"
        print(line)


## 4. Evaluate Main Output

In [7]:
if os.path.exists(OUTPUT_PATH):
    summary_main = evaluate_output(OUTPUT_PATH, benchmark)
    print_summary("Main output", summary_main)
else:
    print(f"[SKIP] {OUTPUT_PATH} not found — run inference.ipynb first.")
    summary_main = None



  Main output
  Queries evaluated  : 72 / 72
  Avg response length: 37 words

  ── Truncated response metrics (primary) ──────────────────────
  Token Recall       : 0.5174   ← most reliable for verbose output
  Token F1           : 0.4781
  Token Precision    : 0.5023
  ROUGE-1 F / R      : 0.4916 / 0.5359
  ROUGE-L F / R      : 0.4315 / 0.4646

  ── Full response metrics (for reference) ─────────────────────
  Token F1 (full)    : 0.4742  (lower due to verbosity)

  By source:
    Around the World in 80 Cuisines Blog     n= 18  Recall=0.4851  F1=0.4536  R1=0.4670
    Wikibooks                                n= 19  Recall=0.5009  F1=0.4659  R1=0.4771
    Wikipedia                                n= 35  Recall=0.5429  F1=0.4973  R1=0.5121


## 5. BERTScore (Optional — GPU recommended)

In [8]:
def add_bertscore(summary: dict, model_type: str = BERTSCORE_MODEL) -> dict:
    """Add BERTScore to an existing summary dict. Modifies in-place."""    
    if not BERTSCORE_AVAILABLE:
        print("[SKIP] Install bert-score first: pip install bert-score")
        return summary

    preds = [e["response"]  for e in summary["per_query"]]
    refs  = [e["reference"] for e in summary["per_query"]]
    print(f"Computing BERTScore for {len(preds)} pairs (model={model_type})...")

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    P, R, F = _bert_score(preds, refs, model_type=model_type,
                           device=device, verbose=False)

    for i, entry in enumerate(summary["per_query"]):
        entry["bertscore_f"] = round(F[i].item(), 4)
        entry["bertscore_p"] = round(P[i].item(), 4)
        entry["bertscore_r"] = round(R[i].item(), 4)

    n = len(preds)
    summary["aggregate"]["bertscore_f"] = round(sum(e["bertscore_f"] for e in summary["per_query"]) / n, 4)
    summary["aggregate"]["bertscore_p"] = round(sum(e["bertscore_p"] for e in summary["per_query"]) / n, 4)
    summary["aggregate"]["bertscore_r"] = round(sum(e["bertscore_r"] for e in summary["per_query"]) / n, 4)

    by_src = defaultdict(list)
    for e in summary["per_query"]:
        by_src[e["source"]].append(e["bertscore_f"])
    for src, vals in by_src.items():
        summary["by_source"][src]["bertscore_f"] = round(sum(vals)/len(vals), 4)

    bs = summary["aggregate"]["bertscore_f"]
    print(f"BERTScore F1={bs:.4f}  "
          f"P={summary['aggregate']['bertscore_p']:.4f}  "
          f"R={summary['aggregate']['bertscore_r']:.4f}")
    print()
    if bs >= 0.70:
        print("  ✓ BERTScore ≥ 0.70 — reasonable for 0.5B model on domain-specific RAG")
    elif bs >= 0.60:
        print("  ~ BERTScore 0.60–0.70 — moderate semantic alignment")
    else:
        print("  ✗ BERTScore < 0.60 — check retrieval quality and prompt design")
    return summary


if summary_main:
    summary_main = add_bertscore(summary_main)


Computing BERTScore for 72 pairs (model=distilbert-base-uncased)...


d:\app\miniconda\envs\rag\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\15813\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 655.16it/s, Materializing param=transformer.layer.5.sa_layer_nor

BERTScore F1=0.8626  P=0.8521  R=0.8759

  ✓ BERTScore ≥ 0.70 — reasonable for 0.5B model on domain-specific RAG


## 6. Batch Evaluate Exploration Outputs

In [9]:
all_summaries = {}
if summary_main:
    all_summaries["main"] = summary_main

if EXPLORATION_DIR and os.path.isdir(EXPLORATION_DIR):
    exp_files = sorted(Path(EXPLORATION_DIR).glob("output_*.json"))
    print(f"Found {len(exp_files)} exploration files in {EXPLORATION_DIR}\n")
    for exp_path in exp_files:
        tag = exp_path.stem.replace("output_", "")
        try:
            s = evaluate_output(str(exp_path), benchmark)
            if s:
                all_summaries[tag] = s
                agg = s["aggregate"]
                print(f"  {tag:35s}  Recall={agg['recall']:.4f}  F1={agg['f1']:.4f}", end="")
                if ROUGE_AVAILABLE:
                    print(f"  RL_R={agg.get('rougeL_r',0):.4f}", end="")
                print()
        except Exception as e:
            print(f"  {tag}: ERROR — {e}")
else:
    print(f"[SKIP] Exploration dir not found: {EXPLORATION_DIR}")

print(f"\nTotal configs evaluated: {len(all_summaries)}")


Found 20 exploration files in results/exploration

  basic_greedy                         Recall=0.5221  F1=0.4982  RL_R=0.4749
  basic_t0.3_p0.9_k50                  Recall=0.5085  F1=0.4941  RL_R=0.4569
  basic_t0.7_p0.95_k100                Recall=0.4685  F1=0.4429  RL_R=0.4262
  basic_t0.7_p0.9_k50                  Recall=0.5174  F1=0.4845  RL_R=0.4698
  basic_t1.0_p0.8_k30                  Recall=0.5260  F1=0.4597  RL_R=0.4823
  cot_greedy                           Recall=0.5358  F1=0.4549  RL_R=0.4930
  cot_t0.3_p0.9_k50                    Recall=0.5277  F1=0.4581  RL_R=0.4955
  cot_t0.7_p0.95_k100                  Recall=0.5121  F1=0.4117  RL_R=0.4672
  cot_t0.7_p0.9_k50                    Recall=0.5175  F1=0.4284  RL_R=0.4634
  cot_t1.0_p0.8_k30                    Recall=0.4901  F1=0.3988  RL_R=0.4494
  enhanced_greedy                      Recall=0.5323  F1=0.4953  RL_R=0.4869
  enhanced_t0.3_p0.9_k50               Recall=0.5278  F1=0.4931  RL_R=0.4857
  enhanced_t0.7_p0.95_k10

## 7. Leaderboard

Ranked by **Token Recall** (primary) then **ROUGE-L Recall** (secondary).
These metrics are more meaningful than F1 for verbose generative models.

In [10]:
if all_summaries:
    rows = []
    for tag, s in all_summaries.items():
        agg = s["aggregate"]
        rows.append({
            "tag":      tag,
            "recall":   agg.get("recall",   0),
            "f1":       agg.get("f1",       0),
            "rougeL_r": agg.get("rougeL_r", 0),
            "rougeL_f": agg.get("rougeL_f", 0),
            "rouge1_f": agg.get("rouge1_f", 0),
            "bs_f":     agg.get("bertscore_f", None),
            "words":    agg.get("pred_words", 0),
        })

    # Sort by recall (primary), rougeL_r (secondary)
    rows.sort(key=lambda x: (x["recall"], x["rougeL_r"]), reverse=True)

    W = 35
    header = (f"{'Config':{W}s}  Recall  F1      RL_R    RL_F    R1_F    "
              + ("BS_F    " if any(r["bs_f"] for r in rows) else "")
              + "Words")
    print(f"\n{'#'*90}")
    print("Leaderboard  (sorted by Token Recall — best metric for verbose generation)")
    print(f"{'#'*90}")
    print(header)
    print("-" * len(header))
    for r in rows:
        line = (f"{r['tag']:{W}s}  "
                f"{r['recall']:.4f}  {r['f1']:.4f}  "
                f"{r['rougeL_r']:.4f}  {r['rougeL_f']:.4f}  {r['rouge1_f']:.4f}  ")
        if any(row["bs_f"] for row in rows):
            line += f"{r['bs_f']:.4f}  " if r["bs_f"] else "        "
        line += f"{r['words']:.0f}"
        print(line)



##########################################################################################
Leaderboard  (sorted by Token Recall — best metric for verbose generation)
##########################################################################################
Config                               Recall  F1      RL_R    RL_F    R1_F    BS_F    Words
------------------------------------------------------------------------------------------
cot_greedy                           0.5358  0.4549  0.4930  0.4192  0.4723          39
enhanced_greedy                      0.5323  0.4953  0.4869  0.4555  0.5093          38
few_shot_greedy                      0.5288  0.3817  0.4635  0.3366  0.3986          63
enhanced_t0.3_p0.9_k50               0.5278  0.4931  0.4857  0.4547  0.5062          36
cot_t0.3_p0.9_k50                    0.5277  0.4581  0.4955  0.4301  0.4771          36
basic_t1.0_p0.8_k30                  0.5260  0.4597  0.4823  0.4192  0.4777          42
few_shot_t0.3_p0.9_k50          

## 8. Save Results

In [15]:
if all_summaries:
    compact = {}
    for tag, s in all_summaries.items():
        compact[tag] = {
            "output_path": s["output_path"],
            "n_evaluated": s["n_evaluated"],
            "aggregate":   s["aggregate"],
            "by_source":   s["by_source"],
        }
    os.makedirs(os.path.dirname(EVAL_OUTPUT) or ".", exist_ok=True)
    with open(EVAL_OUTPUT, "w", encoding="utf-8") as f:
        json.dump(compact, f, ensure_ascii=False, indent=2)
    print(f"Saved summary → {EVAL_OUTPUT}")

    # Save per-query detail for best config
    best = rows[0]["tag"] if rows else None
    if best and best in all_summaries:
        detail_path = EVAL_OUTPUT.replace(".json", f"_detail_{best}.json")
        with open(detail_path, "w", encoding="utf-8") as f:
            json.dump(all_summaries[best], f, ensure_ascii=False, indent=2)
        print(f"Saved detail  → {detail_path}")


Saved summary → results/generation_evaluation_summary.json
Saved detail  → results/generation_evaluation_summary_detail_basic_greedy.json


## 9. Manual Review Helper

Sample low-scoring queries for the **human evaluation** dimensions:

| Dimension | Description | Scale |
|-----------|-------------|-------|
| Correctness | Is the answer factually accurate? | 0–3 |
| Faithfulness | Is it grounded in the retrieved context? | 0–3 |
| Completeness | Does it cover all necessary information? | 0–3 |
| Clarity | Is the phrasing clear and readable? | 0–3 |


In [16]:
def sample_for_review(summary: dict, metric: str = "recall",
                      n: int = 10, worst: bool = True) -> None:
    if not summary or "per_query" not in summary:
        print("No per-query data."); return
    entries = sorted(summary["per_query"],
                     key=lambda e: e.get(metric, 0),
                     reverse=not worst)
    label = "LOWEST" if worst else "HIGHEST"
    print(f"\n{'='*80}")
    print(f"  {label} {n} queries by {metric.upper()}")
    print(f"{'='*80}")
    for i, e in enumerate(entries[:n], 1):
        sc = e.get(metric, 0)
        bs = f"  BS={e['bertscore_f']:.3f}" if "bertscore_f" in e else ""
        print(f"\n[{i}] {e['query_id']}  {metric}={sc:.4f}{bs}  source={e['source']}")
        print(f"  Q:    {e['query']}")
        print(f"  Gold: {e['reference']}")
        resp = e['response']
        print(f"  Pred: {resp[:250]}" + ("..." if len(resp) > 250 else ""))
        print(f"  ── Manual scores ────────────────────────────────────────────")
        print(f"     Correctness (0-3): ___   Faithfulness (0-3): ___")
        print(f"     Completeness(0-3): ___   Clarity      (0-3): ___")

if summary_main:
    sample_for_review(summary_main, metric="recall", n=5, worst=True)



  LOWEST 5 queries by RECALL

[1] q_020  recall=0.1538  BS=0.773  source=Wikipedia
  Q:    How is Dongpo pork prepared?
  Gold: Dongpo pork is prepared by first pan-frying and then slow braising pork belly. The pork is cut into thick squares with an even distribution of fat and lean meat, retaining the skin. The result is tender and juicy with a fragrant aroma of wine.
  Pred: To prepare Dongpo pork, you need to follow these steps:

1. **Preparation**: Cut the pork into thin, square pieces about 5 cm in size. This ensures they are evenly distributed and retain their natural shape when cooked.

2. **Marination**: Marinate t...
  ── Manual scores ────────────────────────────────────────────
     Correctness (0-3): ___   Faithfulness (0-3): ___
     Completeness(0-3): ___   Clarity      (0-3): ___

[2] q_051  recall=0.1622  BS=0.749  source=Wikibooks
  Q:    How is bubble tea assembled?
  Gold: Cooked boba tapioca pearls are placed in a large glass. The remaining ingredients — crushed ic

## 10. Score Distributions

In [17]:
def plot_dist(summary: dict, metric: str) -> None:
    if not summary or "per_query" not in summary:
        return
    scores = [e.get(metric, 0) for e in summary["per_query"]]
    buckets = [0] * 10
    for s in scores:
        buckets[min(int(s * 10), 9)] += 1
    mean   = sum(scores) / len(scores)
    median = sorted(scores)[len(scores)//2]
    print(f"\n{metric.upper()} distribution  "
          f"(n={len(scores)}  mean={mean:.4f}  median={median:.4f})")
    print("-" * 55)
    for i, cnt in enumerate(buckets):
        bar = "█" * cnt
        print(f"  {i*0.1:.1f}–{(i+1)*0.1:.1f}  {bar:<45s} {cnt}")

if summary_main:
    plot_dist(summary_main, "recall")   # primary metric
    plot_dist(summary_main, "f1")       # secondary
    if ROUGE_AVAILABLE:
        plot_dist(summary_main, "rougeL_r")
    if "bertscore_f" in (summary_main["per_query"][0] if summary_main["per_query"] else {}):
        plot_dist(summary_main, "bertscore_f")



RECALL distribution  (n=72  mean=0.5397  median=0.5294)
-------------------------------------------------------
  0.0–0.1                                                0
  0.1–0.2  ████                                          4
  0.2–0.3  ██████████                                    10
  0.3–0.4  ████████                                      8
  0.4–0.5  ███████████                                   11
  0.5–0.6  ████████████                                  12
  0.6–0.7  █████                                         5
  0.7–0.8  ███████████                                   11
  0.8–0.9  █████                                         5
  0.9–1.0  ██████                                        6

F1 distribution  (n=72  mean=0.3650  median=0.3333)
-------------------------------------------------------
  0.0–0.1  █                                             1
  0.1–0.2  ████████                                      8
  0.2–0.3  ████████████████████                          20
  0.3–

## 11. Response Length Analysis

Understand how verbose the model is and whether verbosity correlates with quality.

In [18]:
if summary_main and summary_main.get("per_query"):
    entries = summary_main["per_query"]

    pred_lens = [e["pred_words"] for e in entries]
    ref_lens  = [e["ref_words"]  for e in entries]
    ratios    = [p/r if r > 0 else 0 for p, r in zip(pred_lens, ref_lens)]

    print(f"Response length analysis (n={len(entries)}):")
    print(f"  Avg prediction length : {sum(pred_lens)/len(pred_lens):.1f} words")
    print(f"  Avg reference length  : {sum(ref_lens)/len(ref_lens):.1f} words")
    print(f"  Avg pred/ref ratio    : {sum(ratios)/len(ratios):.1f}x  "
          f"(1.0 = same length; >1 = model is more verbose)")
    print()

    # Bucket by length ratio and show average recall per bucket
    buckets = {"<2x": [], "2–5x": [], "5–10x": [], ">10x": []}
    for e, ratio in zip(entries, ratios):
        if   ratio < 2:   buckets["<2x"].append(e["recall"])
        elif ratio < 5:   buckets["2–5x"].append(e["recall"])
        elif ratio < 10:  buckets["5–10x"].append(e["recall"])
        else:             buckets[">10x"].append(e["recall"])

    print("  Avg Recall by pred/ref length ratio:")
    for label, vals in buckets.items():
        if vals:
            print(f"    {label:8s}: recall={sum(vals)/len(vals):.4f}  (n={len(vals)})")


Response length analysis (n=72):
  Avg prediction length : 94.1 words
  Avg reference length  : 24.3 words
  Avg pred/ref ratio    : 4.0x  (1.0 = same length; >1 = model is more verbose)

  Avg Recall by pred/ref length ratio:
    <2x     : recall=0.6033  (n=16)
    2–5x    : recall=0.5351  (n=36)
    5–10x   : recall=0.5088  (n=18)
    >10x    : recall=0.3929  (n=2)
